In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

## Datasets & Dataloaders

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms  

# image => scale(0,1) => normalize (-1, 1)
transform = transforms.Compose([     #give list of tranformation operations to happen
    transforms.ToTensor(),    # convert image into pytorch tensor and scale them
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # set value to pass on this dataset
])

trainset = CIFAR10(root = "./data", train=True, download=True, transform=transform)
testset = CIFAR10(root = "./data", train=False, download=True, transform=transform)

In [3]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

## Build the CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # first (convo layer)
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # kernel size = 2, stride = 2
        
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # kernel size = 2, stride = 2
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # kernel size = 2, stride = 2
        )

        # second (fully connected layer)
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # flattening
        x = self.fc_layers(x)

        return x

In [5]:
model = CNN()

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training the CNN

In [7]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)  # Forward Prop
        loss = criterion(output, labels)  # loss fnx
        loss.backward()  # Back prop
        optimizer.step()  # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/ {epochs} & loss= {epoch_training_loss/ len(trainloader)}")

epoch=1/ 10 & loss= 1.376427746337393
epoch=2/ 10 & loss= 0.9535542826747041
epoch=3/ 10 & loss= 0.7618241610048372
epoch=4/ 10 & loss= 0.6360344316267297
epoch=5/ 10 & loss= 0.5284523948302964
epoch=6/ 10 & loss= 0.43873397199927694
epoch=7/ 10 & loss= 0.35368058818113773
epoch=8/ 10 & loss= 0.28113135530630035
epoch=9/ 10 & loss= 0.2147347923854123
epoch=10/ 10 & loss= 0.1718060213672307


## Evaluation of CNN

In [8]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        output = model.forward(images)
        _,predicted = torch.max(output, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels / total_labels * 100}")

Accuracy = 74.64
